# 05 — Run the whole pipeline

Owner: the whole group. **This is the notebook to open on Google Colab (CPU runtime).** It loads
notebooks 00–04 with `%run -i` (definitions only, `RUN_TEST = False`) and then runs:

1. **Discovery** (`01`): `discover_all()` builds or resumes `state/discovery_manifest.json`.
2. **Sources** (`02`): every source version, one connection to arXiv, `ARXIV_DELAY` apart (at least 3 s),
   only `.tex`/`.bib` kept. **References** (`03`, Semantic Scholar batch endpoint, 100 papers per request)
   run at the same time in one background thread; the two rate limits are separate.
3. **Report** (`04`): imports `references_export.zip` if present, writes `logs/papers.csv`
   (download columns + reference counts),
   `logs/performance.json` and prints the Milestone 1 statistics.

**Resume.** After a Colab disconnect, run all cells again. Discovery batches, download receipts and
`references.json` files are reused, so only unfinished work is requested again.

**Time.** About 3 s × the number of source versions (roughly a day for 20,000 IDs), so expect 2–3
Colab sessions. `logs/performance.json` describes the latest session; every session is appended to
`logs/performance_history.jsonl`, and the downloader keeps its own `state/download/run_history.jsonl`.

## 0. Setup

Choose which stages to run. References use the Semantic Scholar batch endpoint
(100 papers per request), so they run here in a background thread and finish long before the download.
If a teammate runs `07_run_references.ipynb` instead, set `RUN_REFERENCES = False`; the report step
imports their `references_export.zip`. Every stage resumes from saved state, so leaving all of them on is
safe after a disconnect. `TEST_VERSIONS` (set by `06_test_100_versions.ipynb`) switches to a test
run: only the first assigned IDs, up to that many source versions, written to
`test_runs/<N>_versions/` so the real crawl folders are not touched. `ARXIV_CONTACT` (optional, an email address) is added to the User-Agent.

In [ ]:
# On Colab: clone or update branch VA, install packages and work from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB and not Path("00_config.ipynb").is_file():
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "VA"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                        "https://github.com/thinhelzan842/intro-ds-arxiv-project.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")

RUN_TEST = False        # True also runs each notebook's own tests (01 and 03 call the live APIs)
RUN_DISCOVERY = True    # 01: build / resume the discovery manifest
RUN_DOWNLOAD = True     # 02: download / resume every source version
RUN_REFERENCES = True   # 03: S2 batch requests in a background thread (False if a teammate runs 07)
TEST_VERSIONS = None    # e.g. 100: test run on the first IDs only (see 06_test_100_versions)

# Another notebook (06) can set PIPELINE_OVERRIDES = {...} before %run -i to change the values above.
globals().update(globals().get("PIPELINE_OVERRIDES", {}))

In [ ]:
get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
if TEST_VERSIONS:
    # Test run: move every folder before 01-04 derive their paths from them.
    TEST_ROOT = PROJECT_ROOT / "test_runs" / f"{TEST_VERSIONS}_versions"
    DATA_DIR, LOG_DIR, STATE_DIR = TEST_ROOT / "data" / "raw", TEST_ROOT / "logs", TEST_ROOT / "state"
    PAPER_LOG = LOG_DIR / "papers.csv"
    for folder in (DATA_DIR, LOG_DIR, STATE_DIR):
        folder.mkdir(parents=True, exist_ok=True)
    print(f"TEST RUN: first {TEST_VERSIONS:,} source versions -> {TEST_ROOT}")

for notebook in ("01_discovery", "02_download", "03_references", "04_monitor"):
    get_ipython().run_line_magic("run", f"-i ./{notebook}.ipynb")

from concurrent.futures import ThreadPoolExecutor
import threading

# Disk usage is measured by walking DATA_DIR on Drive, so sample every 30 s (disk every ~2.5 min).
monitor = PerformanceMonitor(sample_interval=30, data_dir=DATA_DIR)
monitor.start()

## 1. Discovery

Skipped quickly when the manifest is already complete. `discovery_seconds` adds up the API time of every
session, so the report shows the real discovery time and not just this session's.

In [ ]:
def discover_first_versions(target_versions):
    """Test run: discover assigned IDs in order until they hold at least target_versions versions."""
    ids = all_assigned_ids()
    n = max(len((read_json(MANIFEST) or {"papers": []})["papers"]), ARXIV_BATCH_SIZE)
    any_query = False
    while True:
        found_papers, queried, seconds = discover_all(ids[:n])
        any_query = any_query or queried
        if sum(p["versions"] for p in found_papers) >= target_versions or n >= len(ids):
            break
        n += ARXIV_BATCH_SIZE
    sample, total = [], 0
    for paper in found_papers:
        if total >= target_versions:
            break
        sample.append(paper)
        total += paper["versions"]
    return sample, any_query, seconds


did_query = False
if TEST_VERSIONS:
    monitor.start_stage("discovery")
    papers, did_query, discovery_seconds = discover_first_versions(TEST_VERSIONS)
    monitor.end_stage("discovery")
    validate_manifest(papers)
    DOWNLOAD_MANIFEST = STATE_DIR / "test_manifest.json"
    write_json(DOWNLOAD_MANIFEST, {"papers": papers})
elif RUN_DISCOVERY:
    monitor.start_stage("discovery")
    papers, did_query, discovery_seconds = discover_all()
    monitor.end_stage("discovery")
else:
    saved = read_json(MANIFEST)
    if saved is None:
        raise FileNotFoundError(f"{MANIFEST} is missing: run with RUN_DISCOVERY = True first")
    papers, discovery_seconds = saved["papers"], saved.get("discovery_seconds", 0.0)

if not TEST_VERSIONS:
    validate_assigned_manifest(papers)
    DOWNLOAD_MANIFEST = MANIFEST
if did_query:
    time.sleep(ARXIV_DELAY)   # keep the API-to-source transition at least 3 s apart
monitor.stages["discovery"] = round(discovery_seconds, 3)   # API time over all sessions, for the report
found = [p["id"] for p in papers if p.get("versions", 0) > 0]   # IDs that exist on arXiv
print(f"{len(papers):,} IDs, {len(found):,} found on arXiv, "
      f"{sum(p.get('versions', 0) for p in papers):,} source versions")

## 2. Sources and references

`download_sources` blocks this cell for the whole crawl and prints progress every 100 papers.
Stopping the cell (or a disconnect) is safe: archives already fetched are processed first, and the
references thread stops after its current batch request.

In [ ]:
stop_references = threading.Event()


def fetch_missing_references(ids):
    """crawl_references (S2 batch endpoint, S2_BATCH_SIZE papers per request); stops when asked.

    Finished papers (ok / not_found) are skipped, so a restart only asks for what is missing.
    """
    fetched = 0
    for start in range(0, len(ids), S2_BATCH_SIZE):
        if stop_references.is_set():
            break
        fetched += len(crawl_references(ids[start:start + S2_BATCH_SIZE], verbose=False))
        if (start // S2_BATCH_SIZE + 1) % 20 == 0:
            print(f"[references] {min(start + S2_BATCH_SIZE, len(ids)):,}/{len(ids):,} IDs checked")
    return fetched


references_pool = references_future = download_summary = None
try:
    if RUN_REFERENCES:
        monitor.start_stage("references")
        references_pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix="references")
        references_future = references_pool.submit(fetch_missing_references, found)
        references_future.add_done_callback(lambda _: monitor.end_stage("references"))
    if RUN_DOWNLOAD:
        monitor.start_stage("download")
        download_summary = download_sources(DOWNLOAD_MANIFEST, DATA_DIR, DOWNLOAD_STATE, user_agent=DOWNLOAD_USER_AGENT,
                                            workers=DOWNLOAD_WORKERS, verify=DOWNLOAD_VERIFY)
        monitor.end_stage("download")
        if download_summary["interrupted"]:
            stop_references.set()
    if references_future is not None:
        if not stop_references.is_set():
            print("[references] download finished; waiting for the remaining references ...")
        print(f"[references] {references_future.result():,} papers fetched in this session "
              f"({REFERENCE_STATS['requests']:,} requests, {REFERENCE_STATS['rate_limited']:,} HTTP 429)")
except KeyboardInterrupt:
    print("Interrupted; saved work is kept and the next run resumes from it.")
finally:
    stop_references.set()
    if references_pool is not None:
        references_pool.shutdown(wait=True)

## 3. Paper log and statistics

If `references_export.zip` from `07_run_references.ipynb` is in the project folder (next to `data/`),
its `references.json` files and `references.csv` are imported first. A saved `ok` file is never
replaced by an `error` one, so importing the same or a newer zip again is safe.

`logs/papers.csv` (`LOG_FIELDS` from `00_config`) is rebuilt from the downloader's `state/download/papers.csv`
plus the reference counts in each `references.json`, so it is the same however many sessions the crawl took.
`04_monitor.ipynb` and the Jupyter Book report read this file.

In [ ]:
REFERENCES_EXPORT = PROJECT_ROOT / "references_export.zip"   # written by 07_run_references.ipynb
_REFERENCE_MEMBER = re.compile(r"^data/raw/([A-Za-z0-9._-]+)/references\.json$")


def import_references(zip_path):
    """Copy references.json files and references.csv from a 07 export into DATA_DIR / LOG_DIR."""
    imported = kept = 0
    with zipfile.ZipFile(zip_path) as package:
        for name in package.namelist():
            match = _REFERENCE_MEMBER.match(name)
            if not match:
                continue
            doc = json.loads(package.read(name))
            target = DATA_DIR / match.group(1) / "references.json"
            previous = read_json(target)
            if previous and previous.get("status") == "ok" and doc.get("status") != "ok":
                kept += 1   # never replace a good result with an error
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            write_json(target, doc)
            imported += 1
        if "logs/references.csv" in package.namelist():
            log_bytes = package.read("logs/references.csv")
            (LOG_DIR / "references.csv").write_bytes(log_bytes)   # the 07 log replaces any older copy
    return imported, kept


def write_paper_log(download_csv=None, path=None):
    """logs/papers.csv for 04_monitor: download columns from 02 + reference counts from 03."""
    download_csv = Path(download_csv or DOWNLOAD_STATE / "papers.csv")
    path = Path(path or PAPER_LOG)
    with download_csv.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    for row in rows:
        refs = load_references(row["id"])
        if refs and refs.get("status") == "ok":
            row["n_refs"], row["n_arxiv_refs"] = refs["n_refs"], refs["n_arxiv_refs"]
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=LOG_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)
    return len(rows)


if REFERENCES_EXPORT.is_file() and not TEST_VERSIONS:
    imported, kept = import_references(REFERENCES_EXPORT)
    print(f"Imported {imported:,} references.json files from {REFERENCES_EXPORT.name}"
          + (f" ({kept:,} older ok files kept)" if kept else ""))

monitor.stop()
performance = monitor.save_summary()
with (LOG_DIR / "performance_history.jsonl").open("a", encoding="utf-8") as stream:
    stream.write(json.dumps(performance, ensure_ascii=False) + "\n")

if (DOWNLOAD_STATE / "papers.csv").is_file():
    print(f"logs/papers.csv: {write_paper_log():,} papers")
    stats = compute_scraping_statistics(load_papers_dataframe(), performance,
                                        total_assigned_ids=len(papers))
    display(format_statistics_table(stats))
else:
    print("No download yet: run with RUN_DOWNLOAD = True to create state/download/papers.csv")